In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Document Classification Evaluation

> **Note:** You can customize configuration parameters (such as `IMAGE_PATHS`, `IMAGE_PREFIX`, `GEMINI_PROJECT_ID`, and `GEMINI_LOCATION`) by editing the local `.env` file.

Evaluates Gemini model performance on document classification tasks using full dataset samples and target class subsets.

By default, evaluation uses sample dataset images referenced in:
`gs://github-repo/generative-ai/gemini/use-cases/entity-extraction/images.csv`

## 1. Environment Setup & Configuration
Imports required libraries, reloads modules, loads environment configuration from `.env`, and sets evaluation parameters.

In [ ]:
import os
import sys

# Clone repository and navigate to the entity-extraction folder if running in Colab
if "google.colab" in sys.modules:
    !git clone https://github.com/arieljassan/generative-ai.git
    %cd generative-ai/gemini/use-cases/entity-extraction
    !pip install -q -r requirements.txt


In [ ]:
import os
import sys

from google.colab import auth

# Authenticate your Google Cloud account in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()

In [ ]:
# Set environment variables, including Project ID and location for running Gemini.

%%writefile .env
# Required: Google Cloud Project ID for Vertex AI Gemini API
GEMINI_PROJECT_ID="my-actual-gcp-project-id"
GEMINI_LOCATION="us-central1"

# Application configuration path
CONFIG_PATH="config.json"

# Evaluation dataset paths (used in evaluate.ipynb)
IMAGE_PATHS="gs://github-repo/generative-ai/gemini/use-cases/entity-extraction/images.csv"
IMAGE_PREFIX="gs://github-repo/generative-ai/gemini/use-cases/entity-extraction"
EVAL_DEST="gs://path/to/evaluations"

# Cloud Run deployment settings (Optional)
CLOUD_RUN_PROJECT_ID="my-actual-gcp-project-id"
CLOUD_RUN_REGION="us-central1"
SERVICE_NAME="entity-extraction-service"
PORT=8080

In [ ]:
import dotenv
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Custom modules for document processing pipeline and evaluation logic
import document_processing
import evaluate

# Load environment configuration from local .env file
dotenv.load_dotenv(dotenv_path=".env", override=True)

# Required: Google Cloud project ID hosting the Gemini model endpoint
PROJECT_ID = os.environ.get("GEMINI_PROJECT_ID")
if not PROJECT_ID:
    raise ValueError("GEMINI_PROJECT_ID environment variable must be set.")

# Optional location setting for Gemini API
LOCATION = os.environ.get("GEMINI_LOCATION", "global")

# Path to CSV containing document metadata and ground truth
IMAGE_PATHS = os.environ.get("IMAGE_PATHS", "")

# GCS or local file path prefix for image assets
IMAGE_PREFIX = os.environ.get("IMAGE_PREFIX", "")

# Target Gemini model variant used for evaluation
EVAL_MODEL = "gemini-2.5-flash"

# Seed for reproducible random sampling and stratification
RANDOM_STATE = 42

## 2. Visualization Utilities
Provides plotting utilities to generate confusion matrix heatmaps sorted by exact match accuracy.

In [ ]:
def plot_confusion_matrix(df, title='Confusion Matrix'):
    """Plots a confusion matrix heatmap ordered by mean match score.

    Args:
        df (pd.DataFrame): Evaluation results containing reference,
            response, and exact_match columns.
        title (str): Title string for the generated heatmap plot.
    """
    # Group by ground-truth reference class and calculate mean match score
    avg_df = df.groupby('reference')['exact_match'].mean().reset_index()

    # Sort ascending so lowest-performing classes appear first
    avg_df = avg_df.sort_values(by='exact_match', ascending=True)

    ordered_classes = avg_df['reference'].tolist()

    # Capture any predicted classes not present in reference set
    extra_preds = [cls for cls in df['response'].unique() if cls not in ordered_classes]
    full_order = ordered_classes + extra_preds

    # Generate cross-tabulation table and re-index for class ordering
    cm = pd.crosstab(df['reference'], df['response'])
    cm_ordered = cm.reindex(index=full_order, columns=full_order, fill_value=0)

    # Render confusion matrix heatmap using Seaborn
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(cm_ordered, annot=True, fmt='d', cmap='Blues')

    # Configure axes formatting for readability
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')
    plt.xticks(rotation=90)
    plt.title(title, pad=20)
    plt.ylabel('Actual (Reference)')
    plt.xlabel('Predicted (Response)')

    plt.show()

## 3. Full Dataset Benchmark Evaluation
Executes the evaluation pipeline across all document classes using a representative dataset sample.

In [ ]:
# Run evaluation for document classification task across the full dataset.
importlib.reload(document_processing)
importlib.reload(evaluate)

# Total number of document samples to evaluate
SAMPLE_SIZE = 120

# Execute evaluation pipeline with class stratification to maintain
# proportional representation across classes
result, df = (
    evaluate.run_evaluation(
        project_id=PROJECT_ID,
        location=LOCATION,
        csv_path=IMAGE_PATHS,
        image_prefix=IMAGE_PREFIX,
        eval_model=EVAL_MODEL,
        sample_size=SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=True,
    )
)

# Display summary metrics (Accuracy, Macro F1, Precision, Recall)
print(result.summary_metrics)

# Visualize confusion matrix for all evaluated classes
plot_confusion_matrix(df, title='Confusion Matrix for All Classes')

## 4. Targeted Class Subset Evaluation
Executes evaluation on a specific subset of high-priority document classes for focused error analysis.

In [ ]:
# Run targeted evaluation focused on specific high-priority document classes.
importlib.reload(document_processing)
importlib.reload(evaluate)

# Reduced sample size scoped for targeted subset analysis
SAMPLE_SIZE = 60

# Define specific target classes of interest for error analysis
target_classes = [
    "budget",
    "specification",
    "form",
    "invoice"
]

# Execute evaluation pipeline filtered by target classes
selected_result, selected_df = (
    evaluate.run_evaluation(
        project_id=PROJECT_ID,
        location=LOCATION,
        csv_path=IMAGE_PATHS,
        image_prefix=IMAGE_PREFIX,
        eval_model=EVAL_MODEL,
        sample_size=SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=True,
        classes=target_classes,
    )
)

# Print metrics summary for the selected subset
print(selected_result.summary_metrics)

# Heatmap plot for target class subset predictions
plot_confusion_matrix(selected_df, title='Confusion Matrix for Selected Classes')